# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named 'HF_TOKEN'."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026


## 1. Two paper findings + my methodology questions

### Finding 1 — Growth prediction

The FlyRank research report states that its growth model correctly identifies growing versus declining pages at about 90% accuracy on unseen pages from the same brands and about 75% on brands it has never seen before.

My methodology question is: how exactly is the growth label constructed, and does the validation design prevent information from the same client or brand from appearing across training and test data?

The distinction between unseen pages from known brands and completely unseen brands matters because pages from the same brand can share structural and behavioral characteristics. I would therefore want to verify that the reported validation setup matches the population implied by each claim.

This is a validation question rather than a challenge to the finding.

### Finding 2 — Zombie recovery

The report states that 65.7K pages had zero traffic in the previous month, 59% subsequently came back, and its recovery model achieved about 99% accuracy on unseen pages from the same brands and 97% on unseen brands.

My methodology question is: how is "recovery" defined and over what future window is the label measured? I would also check whether the validation split keeps the future outcome window completely separate from the features used for prediction.

Because recovery is inherently a future outcome, the timing of the label and the feature window needs to be explicit before interpreting the reported accuracy as predictive performance.

Again, this is a constructive methodology check rather than a judgment about the reported result.

## 2. My model under an honest split (before/after)

The Week-5 model was evaluated using a client-grouped split. For this audit, I will compare that honest grouped evaluation with a simpler random row split.

The random row split allows content from the same client to appear in both training and test sets. This can make the test set less representative of performance on genuinely unseen clients.

The client-grouped split keeps all content from a client in only one side of the split.

Both evaluations use the same Logistic Regression model, the same three February features, the same March zero-click target, and Precision@20 / Precision@50.

In [3]:
# recreate the ML-08 dataset
model_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ) > 0
            THEN
                CAST(
                    SUM(gsc_sum_position) FILTER (
                        WHERE gsc_data_available IS TRUE
                    ) AS DOUBLE
                )
                /
                SUM(gsc_impressions) FILTER (
                    WHERE gsc_data_available IS TRUE
                )
            ELSE NULL
        END AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_days_mar

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_feb,
    f.clicks_feb,
    f.avg_position_feb,
    m.clicks_mar,
    m.available_days_mar,

    CASE
        WHEN m.available_days_mar > 0
             AND COALESCE(m.clicks_mar, 0) = 0
        THEN 1
        ELSE 0
    END AS march_zero_click

FROM feb f

JOIN '{DIM_CONTENT}' c
    ON f.content_hash_id = c.content_hash_id

JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE
    f.impressions_feb >= 100
    AND f.clicks_feb >= 3
    AND f.avg_position_feb IS NOT NULL
    AND c.is_published IS TRUE
""").df()

print("Rows:", len(model_df))
display(model_df.head())

Rows: 29362


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,clicks_mar,available_days_mar,march_zero_click
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4270.0,7.0,5.960187,7.0,31,0
1,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5271.0,4.0,7.125783,6.0,31,0
2,client_73cda7b4e4f265ea,content_a7da352b73b02668,6690.0,19.0,7.226756,13.0,31,0
3,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,4314.0,14.0,6.424200,20.0,31,0
4,client_73cda7b4e4f265ea,content_20403327d8d9374c,2756.0,8.0,6.726415,10.0,31,0


In [4]:
# Before — random row split
FEATURES = [
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top_k = order[:k]
    return np.mean(np.asarray(y_true)[top_k])


train_random, test_random = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df["march_zero_click"]
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),
])

random_model.fit(
    train_random[FEATURES],
    train_random["march_zero_click"]
)

random_scores = random_model.predict_proba(
    test_random[FEATURES]
)[:, 1]

random_p20 = precision_at_k(
    test_random["march_zero_click"],
    random_scores,
    20
)

random_p50 = precision_at_k(
    test_random["march_zero_click"],
    random_scores,
    50
)

print(f"Random row split Precision@20: {random_p20:.3f}")
print(f"Random row split Precision@50: {random_p50:.3f}")

Random row split Precision@20: 0.200
Random row split Precision@50: 0.220


In [5]:
# After — client-grouped split
clients = model_df["client_hash_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_grouped = model_df[
    model_df["client_hash_id"].isin(train_clients)
].copy()

test_grouped = model_df[
    model_df["client_hash_id"].isin(test_clients)
].copy()

grouped_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),
])

grouped_model.fit(
    train_grouped[FEATURES],
    train_grouped["march_zero_click"]
)

grouped_scores = grouped_model.predict_proba(
    test_grouped[FEATURES]
)[:, 1]

grouped_p20 = precision_at_k(
    test_grouped["march_zero_click"],
    grouped_scores,
    20
)

grouped_p50 = precision_at_k(
    test_grouped["march_zero_click"],
    grouped_scores,
    50
)

print(f"Grouped split Precision@20: {grouped_p20:.3f}")
print(f"Grouped split Precision@50: {grouped_p50:.3f}")

print(
    "Client overlap:",
    len(
        set(train_grouped["client_hash_id"])
        &
        set(test_grouped["client_hash_id"])
    )
)

Grouped split Precision@20: 0.350
Grouped split Precision@50: 0.300
Client overlap: 0


In [6]:
# before/after table
validation_comparison = pd.DataFrame([
    {
        "validation": "Random row split",
        "Precision@20": random_p20,
        "Precision@50": random_p50,
    },
    {
        "validation": "Client-grouped split",
        "Precision@20": grouped_p20,
        "Precision@50": grouped_p50,
    },
])

display(validation_comparison)

,validation,Precision@20,Precision@50
0,Random row split,0.20,0.22
1,Client-grouped split,0.35,0.30


### Before / after interpretation

The random row split measured Precision@20 of **0.20** and Precision@50 of **0.22**.

The client-grouped split measured Precision@20 of **0.35** and Precision@50 of **0.30**.

The grouped result is the more appropriate estimate for this audit because content from the same client is kept on one side of the split. The difference between the two measurements shows why validation design can affect the observed performance of a model.

## 3. Leakage audit

The final model feature set contains only:

- `impressions_feb`
- `clicks_feb`
- `avg_position_feb`

These are measured during February, before the March outcome window.

The March outcome fields are used to construct the label and evaluate the model, but they are not model features.

I will explicitly check the feature list for future-window and label-derived fields.

In [7]:
forbidden_columns = [
    "march_zero_click",
    "clicks_mar",
    "available_days_mar",
    "march_zero_click_rate",
    "trend_direction",
    "future_clicks",
    "future_impressions",
    "label",
    "target",
]

found_forbidden = [
    col for col in FEATURES
    if col in forbidden_columns
]

print("Model features:")
for col in FEATURES:
    print("-", col)

print("\nForbidden features found:", found_forbidden)

assert not found_forbidden, (
    f"Leakage detected: {found_forbidden}"
)

print("\nLeakage audit: PASS")

Model features:
- impressions_feb
- clicks_feb
- avg_position_feb

Forbidden features found: []

Leakage audit: PASS


In [8]:
print("Feature window: February 2026")
print("Outcome window: March 2026")

assert "march_zero_click" not in FEATURES
assert "clicks_mar" not in FEATURES

print("Temporal feature check: PASS")

Feature window: February 2026
Outcome window: March 2026
Temporal feature check: PASS


## 4. Claim rewrite

### Original claim

The Logistic Regression model performs better than the Week-4 baseline and can identify pages that will receive zero clicks in March.

### Safer claim

On the measured client-grouped test set, the Logistic Regression model achieved higher Precision@20 and Precision@50 than the Week-4 baseline. The result is an observed performance difference on this dataset and validation split; it does not establish that the model will perform similarly on future clients or future time periods.

The model provides a directional decision-support ranking based on February search-performance signals. It should not be interpreted as a causal explanation of why a page receives zero clicks.

## Self-check

- [x] Every section is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors.
- [x] Two research-paper findings are identified with constructive methodology questions.
- [x] The model is evaluated using both a random row split and a client-grouped split.
- [x] The grouped split keeps clients separated between training and testing.
- [x] Before/after Precision@20 and Precision@50 are shown.
- [x] The final feature set contains only February decision-time features.
- [x] Future-window and label-derived fields are checked explicitly.
- [x] Real model failure examples or validation differences are inspected.
- [x] My strongest claim has been rewritten using observed, measured, directional, and decision-support language.
- [x] No client names, URLs, or private queries are included.
- [x] The notebook runs without errors with Runtime → Run all.
- [x] Committed to my repo under `work/notebooks/w06_validation_audit.ipynb`.